# Unidad 5 — Evaluación de Trayectoria, Resultado y Memoria del Agente

**Proyecto:** Sistema Neuro-Agéntico de Triaje PQRS Bancarias (Ruta 3)  
**Integrantes:** Andres Plazas y Jersson Montenegro  

Siguiendo el patrón de evaluación de la Unidad 5 (`7-evaluacion-de-agentes.ipynb`), en este notebook verificamos:
1. **Memoria conversacional aislada por `thread_id`** (`InMemorySaver`).
2. **Trayectoria (`tool_calls`)** y **Resultado** en 4 casos de prueba representativos:
   - **Caso 1:** Saludo/pregunta general → `[]` (0 herramientas).
   - **Caso 2:** Consulta pura de políticas/SLA → `['consultar_politica_y_reglas_triaje']` (1 herramienta).
   - **Caso 3:** Ticket claro de fraude → `['clasificar_ticket_neuronal', 'consultar_politica_y_reglas_triaje']` (2 herramientas).
   - **Caso 4:** Caso de fallo real de la Unidad 4 (baja confianza) → ambas herramientas + alerta de escalamiento a Nivel 2.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env')

from src.agent_u5 import crear_agente_triaje, invocar_agente
from src.model_u4 import DEFAULT_METADATA_PATH

with Path(DEFAULT_METADATA_PATH).open('r', encoding='utf-8') as f:
    metadatos_u4 = json.load(f)

caso_fallo_u4 = metadatos_u4['error_analysis_case']
agente = crear_agente_triaje()
print('Agente construido correctamente con InMemorySaver y 2 herramientas (@tool).')

Agente construido correctamente con InMemorySaver y 2 herramientas (@tool).


## 1. Verificación de Memoria Conversacional (`thread_id`)

In [2]:
# Turno 1 en hilo 'eval-memoria-1'
r1 = invocar_agente(agente, 'Hola, soy Andres, analista de calidad del banco.', thread_id='eval-memoria-1')
print('[Hilo 1 - Turno 1]:', r1['respuesta'])

# Turno 2 en el MISMO hilo
r2 = invocar_agente(agente, '¿Recuerdas cuál es mi nombre y mi cargo?', thread_id='eval-memoria-1')
print('[Hilo 1 - Turno 2]:', r2['respuesta'])

# Turno 1 en un HILO DISTINTO 'eval-memoria-2'
r3 = invocar_agente(agente, '¿Sabes cuál es mi nombre?', thread_id='eval-memoria-2')
print('[Hilo 2 - Turno 1 (aislado)]:', r3['respuesta'])

[Hilo 1 - Turno 1]: ¡Hola, Andrés! Qué gusto saludarte. Soy tu asistente de triaje inteligente en Banco Digital Boyacá. 

¿En qué puedo apoyarte hoy con los tickets, la clasificación neuronal o las políticas internas del banco?


[Hilo 1 - Turno 2]: ¡Claro que sí, Andrés! Me dijiste que te llamas **Andres** y que trabajas como **analista de calidad** del banco. 

¿En qué te puedo colaborar en este momento?


[Hilo 2 - Turno 1 (aislado)]: ¡Hola! No tengo acceso a tu nombre ni a información personal ya que opero como el Agente de Triaje Inteligente de Banco Digital Boyacá. 

¿En qué puedo ayudarte hoy con respecto a tickets bancarios, clasificaciones o políticas internas del banco?


## 2. Evaluación de Trayectoria (`tool_calls`) y Resultado en 4 Casos

In [3]:
casos_de_prueba = [
    {
        'id': 'Caso 1 (Saludo - 0 tools)',
        'pregunta': 'Hola, buenas tardes. ¿En qué me puedes ayudar?',
        'herramientas_esperadas': [],
        'palabras_clave_resultado': ['triaje', 'ticket', 'política'],
    },
    {
        'id': 'Caso 2 (Solo Normativa - 1 tool)',
        'pregunta': (
            'Sin clasificar ningún ticket, dime cuál es el SLA para Comisiones y Reembolsos '
            'y qué saldo promedio mensual se exige para la exención de cuota de manejo.'
        ),
        'herramientas_esperadas': ['consultar_politica_y_reglas_triaje'],
        'palabras_clave_resultado': ['48', '1.500.000'],
    },
    {
        'id': 'Caso 3 (Ticket Fraude - 2 tools)',
        'pregunta': (
            'Realiza el triaje completo (clasificación y protocolo/SLA) para este ticket de cliente: '
            '"I saw a card payment on my statement that I do not recognise at all, I think my card was stolen."'
        ),
        'herramientas_esperadas': ['clasificar_ticket_neuronal', 'consultar_politica_y_reglas_triaje'],
        'palabras_clave_resultado': ['Fraude', '2 horas'],
    },
    {
        'id': f"Caso 4 (Fallo Real U4 ID {caso_fallo_u4['orig_id']} - 2 tools)",
        'pregunta': (
            'Clasifica con la red neuronal y verifica las reglas de escalamiento y SLA para este ticket: '
            f'"{caso_fallo_u4["text"]}"'
        ),
        'herramientas_esperadas': ['clasificar_ticket_neuronal', 'consultar_politica_y_reglas_triaje'],
        'palabras_clave_resultado': ['Nivel 2', '65'],
    },
]

resultados = []
for idx, caso in enumerate(casos_de_prueba, start=1):
    res = invocar_agente(agente, caso['pregunta'], thread_id=f'eval-caso-{idx}')
    usadas = res['nombres_herramientas']
    esperadas = caso['herramientas_esperadas']

    # Verificación estricta de trayectoria a partir de tool_calls
    trayectoria_ok = usadas == esperadas

    # Verificación de resultado (al menos una palabra clave esperada o todas según el caso)
    texto_low = res['respuesta'].lower()
    if idx == 1:
        resultado_ok = any(k.lower() in texto_low for k in caso['palabras_clave_resultado'])
    else:
        resultado_ok = any(k.lower() in texto_low for k in caso['palabras_clave_resultado'])

    resultados.append({
        'Caso': caso['id'],
        'Esperadas': esperadas,
        'Invocadas (tool_calls)': usadas,
        'Trayectoria OK': trayectoria_ok,
        'Resultado OK': resultado_ok,
        'Respuesta': res['respuesta'],
    })
    print(f"=== {caso['id']} ===")
    print(f"  Esperadas:  {esperadas}")
    print(f"  Invocadas:  {usadas}  -> {'OK' if trayectoria_ok else 'FALLÓ'}")
    print(f"  Resultado:  {'OK' if resultado_ok else 'FALLÓ'}")
    print(f"  Respuesta:  {res['respuesta'][:320]}...\n")

=== Caso 1 (Saludo - 0 tools) ===
  Esperadas:  []
  Invocadas:  []  -> OK
  Resultado:  OK
  Respuesta:  ¡Hola! Buenas tardes. 

Soy el **Agente de Triaje Inteligente de Banco Digital Boyacá**. Puedo ayudarte con:

1. **Clasificación automática de tickets y quejas** de clientes utilizando nuestra red neuronal.
2. **Consulta del manual interno de políticas (RAG)** para conocer tiempos de SLA, prioridades, protocolos y regl...



=== Caso 2 (Solo Normativa - 1 tool) ===
  Esperadas:  ['consultar_politica_y_reglas_triaje']
  Invocadas:  ['consultar_politica_y_reglas_triaje']  -> OK
  Resultado:  OK
  Respuesta:  Con base en el manual interno de políticas de Banco Digital Boyacá para la categoría **Comisiones y Reembolsos**:

* **SLA de atención:** 48 horas hábiles (Prioridad Media).
* **Exención de cuota de manejo:** Se exige un saldo promedio mensual superior a **$1.500.000 COP** para obtener la exención del 100%....



=== Caso 3 (Ticket Fraude - 2 tools) ===
  Esperadas:  ['clasificar_ticket_neuronal', 'consultar_politica_y_reglas_triaje']
  Invocadas:  ['clasificar_ticket_neuronal', 'consultar_politica_y_reglas_triaje']  -> OK
  Resultado:  OK
  Respuesta:  A continuación, presento el resultado del triaje completo para el ticket del cliente:

### 1. Resultado de la Clasificación (Red Neuronal)
* **Categoría predicha:** Fraude y Seguridad
* **Confianza:** 100.0% (Certeza Alta)
* **Escalamiento por baja confianza (<65%):** No requerido

---

### 2. Protocolo, SLA y Reglas d...



=== Caso 4 (Fallo Real U4 ID test_1663 - 2 tools) ===
  Esperadas:  ['clasificar_ticket_neuronal', 'consultar_politica_y_reglas_triaje']
  Invocadas:  ['clasificar_ticket_neuronal', 'consultar_politica_y_reglas_triaje']  -> OK
  Resultado:  OK
  Respuesta:  ### 🔍 Resultado del Triaje Inteligente

* **Categoría Predicha:** Cuenta y Verificación
* **Confianza de la Red Neuronal:** 45.6% (`0.4558`)
* **Nivel de Certeza:** Baja *(Top-3: Cuenta y Verificación: 45.6%, Fraude y Seguridad: 34.1%, Tarjetas y Activación: 19.0%)*

---

### ⚠️ Alerta de Gobernanza y Escalamiento
* **...



In [4]:
df_eval = pd.DataFrame([
    {
        'Caso': r['Caso'],
        'Esperadas': ', '.join(r['Esperadas']) or 'Ninguna',
        'Invocadas': ', '.join(r['Invocadas (tool_calls)']) or 'Ninguna',
        'Trayectoria': 'OK' if r['Trayectoria OK'] else 'FALLÓ',
        'Resultado': 'OK' if r['Resultado OK'] else 'FALLÓ',
    }
    for r in resultados
])
exact_tray = sum(r['Trayectoria OK'] for r in resultados) / len(resultados)
exact_res = sum(r['Resultado OK'] for r in resultados) / len(resultados)

print(df_eval.to_string(index=False))
print(f'\nExactitud de trayectoria: {exact_tray:.0%}')
print(f'Exactitud de resultado:   {exact_res:.0%}')

                                         Caso                                                      Esperadas                                                      Invocadas Trayectoria Resultado
                    Caso 1 (Saludo - 0 tools)                                                        Ninguna                                                        Ninguna          OK        OK
             Caso 2 (Solo Normativa - 1 tool)                             consultar_politica_y_reglas_triaje                             consultar_politica_y_reglas_triaje          OK        OK
             Caso 3 (Ticket Fraude - 2 tools) clasificar_ticket_neuronal, consultar_politica_y_reglas_triaje clasificar_ticket_neuronal, consultar_politica_y_reglas_triaje          OK        OK
Caso 4 (Fallo Real U4 ID test_1663 - 2 tools) clasificar_ticket_neuronal, consultar_politica_y_reglas_triaje clasificar_ticket_neuronal, consultar_politica_y_reglas_triaje          OK        OK

Exactitud de trayectoria: 100